<a href="https://colab.research.google.com/github/Sinrez/PythonProjects/blob/main/DH_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import time

# ================ 8192-БИТНОЕ ПРОСТОЕ ЧИСЛО p ИЗ RFC 3526 https://www.protokols.ru/rfc3526/ ================
P_HEX = (
    "FFFFFFFFFFFFFFFFC90FDAA22168C234C4C6628B80DC1CD1"
    "29024E088A67CC74020BBEA63B139B22514A08798E3404DD"
    "EF9519B3CD3A431B302B0A6DF25F14374FE1356D6D51C245"
    "E485B576625E7EC6F44C42E9A637ED6B0BFF5CB6F406B7ED"
    "EE386BFB5A899FA5AE9F24117C4B1FE649286651ECE45B3D"
    "C2007CB8A163BF0598DA48361C55D39A69163FA8FD24CF5F"
    "83655D23DCA3AD961C62F356208552BB9ED529077096966D"
    "670C354E4ABC9804F1746C08CA18217C32905E462E36CE3B"
    "E39E772C180E86039B2783A2EC07A28FB5C55DF06F4C52C9"
    "DE2BCBF6955817183995497CEA956AE515D2261898FA0510"
    "15728E5A8AAAC42DAD33170D04507A33A85521ABDF1CBA64"
    "ECFB850458DBEF0A8AEA71575D060C7DB3970F85A6E1E4C7"
    "ABF5AE8CDB0933D71E8C94E04A25619DCEE3D2261AD2EE6B"
    "F12FFA06D98A0864D87602733EC86A64521F2B18177B200C"
    "BBE117577A615D6C770988C0BAD946E208E24FA074E5AB31"
    "43DB5BFCE0FD108E4B82D120A92108011A723C12A787E6D7"
    "88719A10BDBA5B2699C327186AF4E23C1A946834B6150BDA"
    "2583E9CA2AD44CE8DBBBC2DB04DE8EF92E8EFC141FBECAA6"
    "287C59474E6BC05D99B2964FA090C3A2233BA186515BE7ED"
    "1F612970CEE2D7AFB81BDD762170481CD0069127D5B05AA9"
    "93B4EA988D8FDDC186FFB7DC90A6C08F4DF435C934028492"
    "36C3FAB4D27C7026C1D4DCB2602646DEC9751E763DBA37BD"
    "F8FF9406AD9E530EE5DB382F413001AEB06A53ED9027D831"
    "179727B0865A8918DA3EDBEBCF9B14ED44CE6CBACED4BB1B"
    "DB7F1447E6CC254B332051512BD7AF426FB8F401378CD2BF"
    "5983CA01C64B92ECF032EA15D1721D03F482D7CE6E74FEF6"
    "D55E702F46980C82B5A84031900B1C9E59E7C97FBEC7E8F3"
    "23A97A7E36CC88BE0F1D45B7FF585AC54BD407B22B4154AA"
    "CC8F6D7EBF48E1D814CC5ED20F8037E0A79715EEF29BE328"
    "06A1D58BB7C5DA76F550AA3D8A1FBFF0EB19CCB1A313D55C"
    "DA56C9EC2EF29632387FE8D76E3C0468043E8F663F4860EE"
    "12BF2D5B0B7474D6E694F91E6DBE115974A3926F12FEE5E4"
    "38777CB6A932DF8CD8BEC4D073B931BA3BC832B68D9DD300"
    "741FA7BF8AFC47ED2576F6936BA424663AAB639C5AE4F568"
    "3423B4742BF1C978238F16CBE39D652DE3FDB8BEFC848AD9"
    "22222E04A4037C0713EB57A81A23F0C73473FC646CEA306B"
    "4BCBC8862F8385DDFA9D4B7FA2C087E879683303ED5BDD3A"
    "062B3CF5B3A278A66D2A13F83F44F82DFF310EE074AB6A36"
    "4597E899A0255DC164F31CC50846851DF9AB48195DED7EA1"
    "B1D510BD7EE74D73FAF36BC31ECFA268359046F4EB879F92"
    "4009438B481C6CD7889A002ED5EE382BC9190DA6FC026E47"
    "9558E4475677E9AA9E3050E2765694DFC81F56E880B96E71"
    "60C980DD98EDD3DFFFFFFFFFFFFFFFFF"
)

p = int(P_HEX, 16)
GENERATORS = [2, 3, 5, 7]
BIT_SIZES = [256, 384, 512]


def mod_exp_with_counters(base, exponent, modulus):
    """Модульное возведение в степень с подсчётом операций"""
    if modulus == 1:
        return 0, 0, 0, 0

    result = 1
    base = base % modulus
    mul_count = 0
    square_count = 0
    bit_count = 0
    #Идёт по битам справа налево
    #Для каждого бита возводит основание в квадрат
    #Если текущий бит = 1 — умножает результат на текущее основание
    while exponent > 0:
        bit_count += 1
        if exponent & 1:
            result = (result * base) % modulus
            mul_count += 1
        exponent >>= 1 #отбрасывание последнего бита
        if exponent > 0:
            base = (base * base) % modulus #переход к следующей степени
            square_count += 1

    return result, mul_count, square_count, bit_count

def generate_dh_secrets(bits=256):
    """Генерация a и b для часов и Коннектора"""
    bytes_n = bits // 8
    a_bytes = os.urandom(bytes_n) #Используем системный urandom, в часах есть аналог
    b_bytes = os.urandom(bytes_n)
    a = int.from_bytes(a_bytes, 'big')
    b = int.from_bytes(b_bytes, 'big')

    if a < 2: # Если число получилось 0 или 1 (вероятность низк: 2 шанса из 2²⁵⁶ ≈ 1.15×10⁻⁷⁷)
        a += 2 # Добавляем 2, чтобы получилось минимум 2 (единица в DH ослабляет безопасность, 0 делает общий секрет нулевым)
    if b < 2: #аналогично
        b += 2
    return a, b

def test_dh_with_generators(bits=256):
    """Тестирование DH с разными генераторами g"""
    print("=" * 80)
    print(f"ТЕСТИРОВАНИЕ DIFFIE-HELLMAN С МОДУЛЕМ 8192 БИТ")
    print(f"Размер секретов a, b: {bits} бит")
    print(f"p = {p.bit_length()} бит")
    print("=" * 80)

    print("\n[1] Генерация секретов a и b...")
    start_gen = time.time()
    a, b = generate_dh_secrets(bits)
    gen_time = time.time() - start_gen
    print(f"    a = 0x{format(a, f'0{bits//4}x')[:32]}... ({a.bit_length()} бит)")
    print(f"    b = 0x{format(b, f'0{bits//4}x')[:32]}... ({b.bit_length()} бит)")
    print(f"    Время генерации: {gen_time:.4f} сек")

    results = []

    for g in GENERATORS:
        print(f"\n{'-' * 80}")
        print(f"ТЕСТ С ГЕНЕРАТОРОМ g = {g}")
        print(f"{'-' * 80}")

        start_a = time.time()
        A, mul_a, sq_a, bits_a = mod_exp_with_counters(g, a, p)
        time_a = time.time() - start_a

        print(f"  A = {g}^{a.bit_length()} бит mod p: {time_a:.4f} сек ({bits_a} итераций)")

        start_b = time.time()
        B, mul_b, sq_b, bits_b = mod_exp_with_counters(g, b, p)
        time_b = time.time() - start_b

        start_s = time.time()
        S, mul_s, sq_s, bits_s = mod_exp_with_counters(B, a, p)
        time_s = time.time() - start_s

        start_check = time.time()
        S_check, _, _, _ = mod_exp_with_counters(A, b, p)
        time_check = time.time() - start_check

        is_valid = (S == S_check)
        total_time = time_a + time_b + time_s

        print(f"  B = {g}^{b.bit_length()} бит mod p: {time_b:.4f} сек")
        print(f"  S = B^a mod p: {time_s:.4f} сек")
        print(f"  Проверка: {'+' if is_valid else '-'}, время: {time_check:.4f} сек")
        print(f"  ⏱ ИТОГО: {total_time:.4f} сек")

        results.append({
            'g': g,
            'time_a': time_a,
            'time_b': time_b,
            'time_s': time_s,
            'total_time': total_time,
            'bits_a': bits_a,
            'valid': is_valid,
            'A': A, 'B': B, 'S': S
        })

    print("\n" + "=" * 100)
    print("СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
    print("=" * 100)
    print(f"{'g':^4} | {'A = g^a (сек)':^14} | {'B = g^b (сек)':^14} | {'S = B^a (сек)':^14} | {'ИТОГО (сек)':^14} | {'Итераций':^10}")
    print("-" * 85)
    for r in results:
        print(f"{r['g']:^4} | {r['time_a']:^14.4f} | {r['time_b']:^14.4f} | {r['time_s']:^14.4f} | {r['total_time']:^14.4f} | {r['bits_a']:^10}")

    print("\n" + "=" * 80)
    print("ФОРМИРОВАНИЕ КЛЮЧА AES.WATCH.DEK")
    print("=" * 80)
    for r in results:
        s_bytes = r['S'].to_bytes((r['S'].bit_length() + 7) // 8, 'big')
        aes_key_bytes = s_bytes[:32] if len(s_bytes) >= 32 else s_bytes.ljust(32, b'\x00')
        aes_key_hex = aes_key_bytes.hex()
        print(f"\n  g = {r['g']}: AES.WATCH.DEK = {aes_key_hex[:32]}...{aes_key_hex[-8:]}")

    return results

def test_all_bit_sizes_all_g():
    """Тестирование для разной битности без колонки ИТОГО в основной таблице"""
    print("\n" + "=" * 120)
    print("ВЛИЯНИЕ БИТНОСТИ a И b НА ВРЕМЯ ВЫЧИСЛЕНИЙ ДЛЯ ВСЕХ g")
    print("=" * 120)

    # Шапка таблицы
    header = f"{'Битность':^10} |"
    for g in GENERATORS:
        header += f" {'g=' + str(g):^37} |"
    print(header)
    print(f"{'-' * 10} |" + "".join([f" {'-' * 37} |" for _ in GENERATORS]))

    # Подшапка: A, B, S без ИТОГО
    subheader = f"{'':^10} |"
    for _ in GENERATORS:
        subheader += f" {'A (сек)':^10} {'B (сек)':^10} {'S (сек)':^12} |"
    print(subheader)
    print(f"{'-' * 10} |" + "".join([f" {'-' * 37} |" for _ in GENERATORS]))

    # Словарь для хранения суммарных значений (для итоговой таблицы)
    totals = {g: [] for g in GENERATORS}

    for bits in BIT_SIZES:
        a, b = generate_dh_secrets(bits)

        row = f"{bits:^10} |"

        for g in GENERATORS:
            start_a = time.time()
            A, _, _, _ = mod_exp_with_counters(g, a, p)
            time_a = time.time() - start_a

            start_b = time.time()
            B, _, _, _ = mod_exp_with_counters(g, b, p)
            time_b = time.time() - start_b

            start_s = time.time()
            S, _, _, _ = mod_exp_with_counters(B, a, p)
            time_s = time.time() - start_s

            total = time_a + time_b + time_s
            totals[g].append((bits, total))

            row += f" {time_a:^10.4f} {time_b:^10.4f} {time_s:^12.4f} |"

        print(row)

    print(f"{'-' * 10} |" + "".join([f" {'-' * 37} |" for _ in GENERATORS]))

    # ================ ДОПОЛНИТЕЛЬНАЯ ТАБЛИЦА С ИТОГО ================
    print("\n" + "=" * 70)
    print("СВОДНАЯ ТАБЛИЦА: ИТОГОВОЕ ВРЕМЯ (A+B+S) В СЕКУНДАХ")
    print("=" * 70)

    # Шапка итоговой таблицы
    print(f"{'Битность':^10} |", end="")
    for g in GENERATORS:
        print(f" {'g=' + str(g):^16} |", end="")
    print()
    print(f"{'-' * 10} |" + "".join([f" {'-' * 16} |" for _ in GENERATORS]))

    # Строки итоговой таблицы
    for i, bits in enumerate(BIT_SIZES):
        print(f"{bits:^10} |", end="")
        for g in GENERATORS:
            total = totals[g][i][1]
            print(f" {total:^16.4f} |", end="")
        print()

    print(f"{'-' * 10} |" + "".join([f" {'-' * 16} |" for _ in GENERATORS]))

    print("\nПримечание: A = g^a mod p, B = g^b mod p, S = B^a mod p (общий секрет)")


if __name__ == '__main__':
    print("\n" + "* " * 20)
    print("DIFFIE-HELLMAN KEY EXCHANGE")
    print("Модуль: 8192 бита (RFC 3526 Group 18) https://www.protokols.ru/rfc3526/")
    print("Алгоритм: бинарное возведение в степень (через сдвиги)")
    print("* " * 20)

    test_dh_with_generators(bits=256)
    test_all_bit_sizes_all_g()


* * * * * * * * * * * * * * * * * * * * 
DIFFIE-HELLMAN KEY EXCHANGE
Модуль: 8192 бита (RFC 3526 Group 18) https://www.protokols.ru/rfc3526/
Алгоритм: бинарное возведение в степень (через сдвиги)
* * * * * * * * * * * * * * * * * * * * 
ТЕСТИРОВАНИЕ DIFFIE-HELLMAN С МОДУЛЕМ 8192 БИТ
Размер секретов a, b: 256 бит
p = 8192 бит

[1] Генерация секретов a и b...
    a = 0x55e84e1038e148bac8e02c761b3ec3a8... (255 бит)
    b = 0x5c5fd2b89ed2708fd81fc5bc1b4178f4... (255 бит)
    Время генерации: 0.0000 сек

--------------------------------------------------------------------------------
ТЕСТ С ГЕНЕРАТОРОМ g = 2
--------------------------------------------------------------------------------
  A = 2^255 бит mod p: 0.0615 сек (255 итераций)
  B = 2^255 бит mod p: 0.0617 сек
  S = B^a mod p: 0.0639 сек
  Проверка: +, время: 0.0658 сек
  ⏱ ИТОГО: 0.1870 сек

--------------------------------------------------------------------------------
ТЕСТ С ГЕНЕРАТОРОМ g = 3
----------------------------------